# صوت الفيديو الترويجي (الأستاذ سوايسية هشام)
شغّل الخلايا بالترتيب (Runtime ← Run all).
- **الخلية 3** تُسمعك جملة واحدة بخمسة أصوات. الصوت المختار مسبقاً: **ar-SA-HamedNeural** (نفس صوت حلقة الدالة الأسية). إن أعجبك غيره ضع اسمه في `VOICE` وأعد التشغيل.
- **الخلية 4** تصنع الصوت كاملاً وتحمّل `promo_voice.zip` (فيه `voice.mp3` و`timings.json`). أرسله كما هو.


In [ ]:
!pip -q install -U edge-tts pydub "audioop-lts; python_version>='3.13'"

In [ ]:
VOICE = "ar-SA-HamedNeural"   # بدائل: ar-EG-ShakirNeural ، ar-AE-HamdanNeural ، ar-JO-TaimNeural ، ar-DZ-IsmaelNeural
RATE = "+4%"     # سرعة إعلانية خفيفة
PITCH = "+0Hz"
GAP = 0.7         # صمت بين الجمل (ثوانٍ)
LEAD = 0.5
LINES = [
 "لكل أستاذ تعليم ثانوي يدرّس في القسم أو في دروس الدعم، ولكل تلميذ يدرس في الثانوية ويحتاج ملخصات الدروس أو حلول التمارين: هذه الحزمة تحتوي على كل ما تحتاجه، بجميع الصيغ.",
 "سلسلة دروس الأستاذ سوايسية هشام، من السنة الأولى إلى السنة الثالثة ثانوي، مطابقة للتوزيع السنوي والمنهاج الجزائري.",
 "لكل درس ثلاثة ملفات: ملخص شامل بالأمثلة والأشكال، وسلسلة تمارين بحلول مفصلة خطوة بخطوة، وتمارين البكالوريا محلولة إلى غاية ألفين وستة وعشرين.",
 "وكل ملف بثلاث صيغ: بي دي إف جاهز للطباعة، ووورد تعدّله كما تشاء، ولاتاك للكتابة الاحترافية.",
 "في السنة الثالثة: الدالة الأسية واللوغاريتمية، النهايات، المتتاليات، التكامل، الاحتمالات، الأعداد المركبة، الحساب، الهندسة في الفضاء، والإحصاء لشعبة تسيير واقتصاد.",
 "وفي السنتين الأولى والثانية: الدوال، كثيرات الحدود، المتتاليات، الاحتمالات، الزوايا الموجهة، والهندسة المستوية... والقائمة تكبر.",
 "جداول تغيّرات، ومنحنيات ملوّنة، وجداول احتمالات منسّقة... كل درس مراجَع ومصحَّح سطراً بسطر.",
 "للحصول على الملفات، تواصل معنا عبر صفحة الأستاذ سوايسية هشام للرياضيات على فيسبوك وتلغرام، وعلى إنستغرام: بروف هشام ماث."
]

In [ ]:
import io, asyncio, edge_tts
from IPython.display import Audio, display
from pydub import AudioSegment
from pydub.silence import detect_leading_silence

async def tts(text, voice=None, rate=None):
    voice, rate = voice or VOICE, rate or RATE
    for attempt in range(5):
        try:
            try:
                com = edge_tts.Communicate(text, voice, rate=rate, pitch=PITCH, boundary="WordBoundary")
            except TypeError:
                com = edge_tts.Communicate(text, voice, rate=rate, pitch=PITCH)
            words, data = [], bytearray()
            async for ch in com.stream():
                if ch["type"] == "audio": data += ch["data"]
                elif ch["type"] == "WordBoundary":
                    words.append({"text": ch["text"], "start": ch["offset"] / 1e7, "end": (ch["offset"] + ch["duration"]) / 1e7})
            seg = AudioSegment.from_file(io.BytesIO(bytes(data)), format="mp3")
            a = max(0, detect_leading_silence(seg, silence_threshold=-45) - 40)
            b = max(0, detect_leading_silence(seg.reverse(), silence_threshold=-45) - 40)
            seg = seg[a:len(seg) - b]
            return seg, [{**w, "start": max(0, w["start"] - a / 1000), "end": max(0, w["end"] - a / 1000)} for w in words]
        except Exception as err:
            print("إعادة المحاولة:", err); await asyncio.sleep(2 + 2 * attempt)
    raise RuntimeError(text)

for v in ["ar-SA-HamedNeural", "ar-EG-ShakirNeural", "ar-AE-HamdanNeural", "ar-JO-TaimNeural", "ar-DZ-IsmaelNeural"]:
    seg, _ = await tts(LINES[0], v)
    seg.export(f"{v}.mp3", format="mp3"); print(v); display(Audio(f"{v}.mp3"))

In [ ]:
import json, zipfile
from google.colab import files
full = AudioSegment.silent(duration=int(LEAD * 1000)); out = []
for i, line in enumerate(LINES):
    seg, words = await tts(line)
    t0 = len(full) / 1000
    out.append({"i": i + 1, "text": line, "start": t0, "end": t0 + len(seg) / 1000,
                "words": [{**w, "start": w["start"] + t0, "end": w["end"] + t0} for w in words]})
    full += seg + AudioSegment.silent(duration=int(GAP * 1000))
    print(f"{i + 1}  {len(seg) / 1000:5.1f}s")
full.export("voice.mp3", format="mp3", bitrate="192k")
json.dump({"voice": VOICE, "rate": RATE, "duration": len(full) / 1000, "lines": out}, open("timings.json", "w"), ensure_ascii=False, indent=1)
with zipfile.ZipFile("promo_voice.zip", "w") as z:
    z.write("voice.mp3"); z.write("timings.json")
print("المدة:", round(len(full) / 1000), "ثانية"); display(Audio("voice.mp3"))
files.download("promo_voice.zip")